# Compact Public-Data-Conditioned Detector-Level 2 m SCAO Demonstrator

This notebook is the fast end-to-end integration path for the compact, public-data-informed 2 m SCAO demonstrator. It reuses the detector SH-WFS calibration, synthetic DM model, detector-level interaction matrix, closed-loop controller, science PSF metrics, error-budget scenarios, and validation checks from the reusable `src/` modules.

Public data (ESO Paranal ASM nighttime atmosphere, SVO 2MASS J/H/Ks filter curves, Pan-STARRS/2MASS photometry) condition selected inputs; the AO internal model (DM, interaction matrix, reconstructor, latency, NCPA, misregistration) stays synthetic or literature-inspired. The result is a compact, inspectable engineering demonstrator. It is **not** calibrated observatory AO telemetry, a measured DM/RTC calibration, or a VLT/Paranal performance prediction.

## 1. Fast reproducibility check

**Purpose:** run the end-to-end detector-level closed loop over the eight-scenario error budget at fast numerical scale.

**Outputs:** the compact `fast_reference_metrics.json` (open/closed OPD RMS, H Strehl, kept modes, 6/6 validation) plus the fast error-budget and validation CSV/PNG artifacts.

**Verification:** the notebook runs top-to-bottom in fast mode without internet, with finite metrics and passing validation checks.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ao_integration import IntegrationConfig, run_fast_integration


In [ ]:
output_dir = Path(os.environ.get("AO_DEMO_OUTPUT_DIR", ROOT / "figures" / "detector_level_SCAO"))
reference_metrics_path = Path(
    os.environ.get(
        "AO_DEMO_REFERENCE_METRICS",
        ROOT / "data" / "reference_metrics" / "fast_reference_metrics.json",
    )
)

config = IntegrationConfig.from_mode(
    "fast",
    output_dir=output_dir,
    reference_metrics_path=reference_metrics_path,
)
result = run_fast_integration(config=config, write_outputs=True)
result.reference_metrics


In [ ]:
assert result.reference_metrics["scenario_count"] == 8
assert result.reference_metrics["validation_pass_count"] == result.reference_metrics["validation_check_count"]
assert result.reference_metrics["kept_modes"] >= 1
assert 0.0 <= result.reference_metrics["h_strehl"] <= 1.5
assert all(path.exists() and path.stat().st_size > 0 for path in result.written_files)
[path.name for path in result.written_files]


Fast mode writes:

- `figures/detector_level_SCAO/fast_error_budget.csv`
- `figures/detector_level_SCAO/fast_error_budget.png`
- `figures/detector_level_SCAO/fast_validation.csv`
- `figures/detector_level_SCAO/fast_validation.png`
- `data/reference_metrics/fast_reference_metrics.json`

The JSON file is intentionally small and includes tolerance bands for future regression checks.

## 2. Public-data-informed portfolio table

**Physical step:** display the public-data-informed scenario table produced by `examples/run_public_data_informed_ao_demo.py` (ESO-ASM-conditioned synthetic phase sequence + Pan-STARRS photon-budget anchor + synthetic AO internal effects).

**Expected output:** a provenance-tagged summary of the five public conditions loaded from the same configured artifact directory. The table is required by the notebook smoke contract; recomputing it remains an optional slower command-line step.

**Covers:** the public-data-informed scenario extension.

Heavier `portfolio`/`research` reruns and the full public-data-informed recompute are intentionally left as optional command-line runs, not part of the fast smoke path.

In [ ]:
# Display the pre-generated public-data-informed scenario table from the same
# configured artifact directory. The slow recompute remains separate/offline.
import csv

public_csv = output_dir / "public_data_informed_error_budget.csv"
assert public_csv.exists(), (
    f"Required public-data-informed table is missing: {public_csv}. "
    "Run `python examples/run_public_data_informed_ao_demo.py` to refresh it."
)
with public_csv.open(newline="", encoding="utf-8") as handle:
    public_rows = list(csv.DictReader(handle))
assert public_rows, "Public-data-informed table must contain at least one row."
assert any(
    "ESO-ASM-conditioned synthetic phase sequence" in row.get("phase_sequence_provenance", "")
    for row in public_rows
), "Public-data table did not load an ESO-ASM-conditioned row."
for row in public_rows:
    print(
        f"{row.get('condition_name', '?'):>20}: "
        f"closed RMS={row.get('closed_rms_nm', 'n/a')} nm, "
        f"H Strehl={row.get('strehl_H', 'n/a')} | "
        f"{row.get('phase_sequence_provenance', '')}"
    )

## Limitations

This demonstrator deliberately excludes, and must not be read as providing:

- private AO telemetry or real RTC latency logs;
- a measured DM influence/interaction matrix or measured detector calibration frames;
- a calibrated VLT/Paranal observatory performance prediction or on-sky PSF validation.

Public data condition the atmosphere (ESO ASM seeing → r0 → ESO-ASM-conditioned synthetic phase sequence), the science bandpasses (SVO 2MASS J/H/Ks), and the WFS photon-budget anchor (Pan-STARRS DR2, an AB-magnitude engineering estimate). All AO internal terms — DM influence functions, interaction matrix, reconstructor, latency decomposition, NCPA, and WFS/DM misregistration — are synthetic or literature-inspired engineering proxies.